# MalImg — Kaggle experiment workspace

Frozen image features → traditional ML → validation → explicit final test.

**Two training engines:** `classical` retains LR/LinearSVC and trial checkpoints;
`sgd_epoch` is a separate SGD experiment with a committed checkpoint after every
epoch, validation history, best-epoch selection, and early stopping.
SGD is not the same optimizer/model run as the old LR/LinearSVC baseline.

**Persistence:** `/kaggle/working` is local session storage. With private Dataset
backup enabled, wait for **REMOTE VERIFIED**. Every epoch uploads a version and
downloads it for checksum verification. Quota/network errors stop before another
epoch; an unfinished epoch restarts from the previous committed epoch.

In Kaggle: enable Internet; choose GPU for feature extraction; add private Secrets
`KAGGLE_USERNAME` and `KAGGLE_KEY` and enable access for this notebook. Never paste
keys into cells. Keep the notebook private while using personal checkpoint inputs.

In [ ]:
# Change these controls, then execute this cell before setup.
ACTION = 'train'  # prepare | extract | train | analyze | final | backup
ENGINE = 'sgd_epoch'  # sgd_epoch | classical
FINAL_TEST = False  # only True together with ACTION='final', after choices are frozen

REPO_URL = 'https://github.com/TuanKhai1210/malimg-family-classification.git'
REPO_REF = 'codex/kaggle-epoch-checkpoints'
WORKSPACE = '/kaggle/working/malimg-family-classification'
WORKING_ROOT = None  # existing writable workspace after a session restart
CONFIG_FILE = 'configs/project.json'
INSTALL_DEPENDENCIES = False  # Kaggle normally has the ML packages; enable if missing
INSTALL_KAGGLE_API = True  # install the reviewed kaggle==2.2.5 adapter if needed

AUTO_BACKUP = True
BACKUP_DATASET = 'yourname/malimg-experiment-checkpoints'  # change username; dedicated PRIVATE dataset
BACKUP_DIR = '/kaggle/working/malimg-backups'  # local backup spool, outside WORKSPACE
RESTORE_REMOTE = False  # True: fetch latest private recovery bundle before importing source
RESTORE_SNAPSHOT = None  # existing snapshots/*.zip + its paired features/ folder (old Drive backup)
RESTORE_PAYLOAD = None  # recovery.zip attached under /kaggle/input (alternative to RESTORE_REMOTE)
RESTORE_TO = '/kaggle/working/malimg-restored'  # must NOT exist on first restore
NEW_RUN_FROM_BACKUP = False  # True to reuse old artifacts with current source; RUN_ID must be None
PIN_RESTORE_LIBRARIES = True  # exact resume: require matching Python, pin serialization packages

RUN_ID = None  # explicit sgd_... or classical validation_...; None starts a preserved NEW run
ARCHIVE_PATH = None  # optional audited original ZIP under /kaggle/input; never True/False
FEATURE_DIRS = {}  # optional {'resnet50': 'features/resnet50/<signature>'} if multiple caches exist
EPOCH_SETTINGS = {
    'extractor': 'resnet50', 'loss': 'log_loss', 'alpha': 0.0001,
    'epochs': 30, 'batch_size': 256, 'patience': 6,
    'learning_rate': 'optimal', 'eta0': 0.01,
}
MAX_SESSION_MINUTES = 360  # planned graceful pause; not a guarantee about Kaggle runtime limits
EPOCHS_THIS_SESSION = None  # optional small segment; keep EPOCH_SETTINGS unchanged for exact resume
SHOW_PLOTS = True
EXPORT_REVIEW_ZIP = True

## 1. Setup and recovery

New run: fill `BACKUP_DATASET`, leave `RUN_ID=None`. Exact resume: set
`RESTORE_REMOTE=True` and your explicit `RUN_ID`; the saved source/config/settings
are used. After restoring once, set `RESTORE_REMOTE=False`,
`RESTORE_SNAPSHOT=None`, `RESTORE_PAYLOAD=None`, `WORKING_ROOT=RESTORE_TO` before
rerunning setup. Do not restore over a populated directory.

To import the old Colab baseline: attach both `snapshots/` and `features/` as a
private input, set `RESTORE_SNAPSHOT`, `NEW_RUN_FROM_BACKUP=True`, `RUN_ID=None`.
This preserves old results and reuses verified fixed features for a **new** SGD
run; it does not turn LR/SVC into epoch checkpoints. Analysis needs no new fit.

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import math
import subprocess
import sys
from os import PathLike

def _validate_controls():
    for name in ('FINAL_TEST', 'AUTO_BACKUP', 'RESTORE_REMOTE', 'NEW_RUN_FROM_BACKUP',
                 'INSTALL_DEPENDENCIES', 'INSTALL_KAGGLE_API', 'PIN_RESTORE_LIBRARIES',
                 'SHOW_PLOTS', 'EXPORT_REVIEW_ZIP'):
        if name not in globals() or type(globals()[name]) is not bool:
            raise TypeError(f'Run the complete controls cell: {name} must be True/False')
    for name, optional in (('WORKSPACE', False), ('WORKING_ROOT', True), ('CONFIG_FILE', False),
                           ('BACKUP_DIR', False), ('RESTORE_SNAPSHOT', True), ('RESTORE_PAYLOAD', True),
                           ('RESTORE_TO', False), ('ARCHIVE_PATH', True), ('RUN_ID', True)):
        if name not in globals():
            raise RuntimeError(f'Run the complete controls cell to define {name}')
        value = globals()[name]
        if value is None and optional:
            continue
        if isinstance(value, bool) or not isinstance(value, (str, PathLike)) or not str(value).strip():
            raise TypeError(f'{name} must be a path/run ID string' + (' or None' if optional else ''))
    if ACTION not in {'prepare', 'extract', 'train', 'analyze', 'final', 'backup'}:
        raise ValueError('Unknown ACTION')
    if ENGINE not in {'classical', 'sgd_epoch'}:
        raise ValueError('Unknown ENGINE')
    if FINAL_TEST != (ACTION == 'final'):
        raise ValueError("FINAL_TEST=True is required only with ACTION='final'; keep it False otherwise")
    if sum((RESTORE_REMOTE, RESTORE_SNAPSHOT is not None, RESTORE_PAYLOAD is not None)) > 1:
        raise ValueError('Choose exactly one recovery source')
    if NEW_RUN_FROM_BACKUP and RUN_ID is not None:
        raise ValueError('Source migration starts a NEW run; RUN_ID must be None')
    if MAX_SESSION_MINUTES is not None and (isinstance(MAX_SESSION_MINUTES, bool) or
            not isinstance(MAX_SESSION_MINUTES, (int, float)) or
            not math.isfinite(MAX_SESSION_MINUTES) or MAX_SESSION_MINUTES <= 0):
        raise ValueError('MAX_SESSION_MINUTES must be positive or None')
    if EPOCHS_THIS_SESSION is not None and (type(EPOCHS_THIS_SESSION) is not int or EPOCHS_THIS_SESSION < 1):
        raise ValueError('EPOCHS_THIS_SESSION must be a positive integer or None')
    if not isinstance(FEATURE_DIRS, dict) or any(isinstance(v, bool) or not isinstance(v, (str, PathLike)) for v in FEATURE_DIRS.values()):
        raise TypeError('FEATURE_DIRS must map extractor names to feature-store paths')

_validate_controls()
needs_api = RESTORE_REMOTE or (AUTO_BACKUP and ACTION != 'analyze')
if needs_api and INSTALL_KAGGLE_API:
    try:
        api_version = importlib.metadata.version('kaggle')
    except importlib.metadata.PackageNotFoundError:
        api_version = None
    if api_version != '2.2.5':
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle==2.2.5'], check=True)

bootstrap = Path(WORKING_ROOT or WORKSPACE).expanduser().resolve()
if not (bootstrap / 'scripts/kaggle_workspace.py').is_file():
    if WORKING_ROOT is not None or bootstrap.exists():
        raise FileNotFoundError('Workspace lacks the Kaggle tools; inspect it or select a NEW WORKSPACE')
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(bootstrap)], check=True)
repo_root = bootstrap
if RESTORE_REMOTE or RESTORE_SNAPSHOT is not None or RESTORE_PAYLOAD is not None:
    command = [sys.executable, str(bootstrap / 'scripts/kaggle_workspace.py'), '--destination', RESTORE_TO]
    if RESTORE_REMOTE:
        command += ['--dataset', BACKUP_DATASET, '--backup-dir', BACKUP_DIR]
    elif RESTORE_SNAPSHOT is not None:
        command += ['--snapshot', str(RESTORE_SNAPSHOT)]
    else:
        command += ['--payload', str(RESTORE_PAYLOAD)]
    if NEW_RUN_FROM_BACKUP:
        command += ['--upgrade-from', str(bootstrap)]
    subprocess.run(command, check=True)
    repo_root = Path(RESTORE_TO).resolve()

for name, module in list(sys.modules.items()):
    if name == 'modules' or name.startswith('modules.'):
        filename = getattr(module, '__file__', None)
        if filename and not Path(filename).resolve().is_relative_to(repo_root):
            raise RuntimeError('Another workspace is imported. Restart Python, then use WORKING_ROOT for the restored folder')

restore_path = repo_root / 'RESTORED_BACKUP.json'
restore_record = json.loads(restore_path.read_text(encoding='utf-8')) if restore_path.is_file() else {}
pins = []
if restore_record and PIN_RESTORE_LIBRARIES and not NEW_RUN_FROM_BACKUP and ACTION != 'analyze':
    if restore_record['python'].split()[0].split('.')[:2] != list(map(str, sys.version_info[:2])):
        raise RuntimeError('Exact resume needs the original Python major/minor; use artifact migration with RUN_ID=None for a NEW run')
    pins = [f'{name}=={version}' for name, version in restore_record['software'].items()
            if name in {'numpy', 'scipy', 'scikit-learn', 'joblib'}]
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo_root / 'requirements.txt'), *pins], check=True)
elif pins:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *pins], check=True)
sys.path.insert(0, str(repo_root))
print('Workspace:', repo_root)
print('SGD runs:', [p.name for p in sorted((repo_root / 'results/epoch_runs').glob('sgd_*'))])
print('Classical runs:', [p.name for p in sorted((repo_root / 'results/runs').glob('validation_*'))])

## 2. Configuration and durable-backup preflight

An explicit `RUN_ID` loads that run's saved configuration. SGD resume also loads
its original settings; changing loss/alpha/seed/preprocessing creates a new run.
Class weights and scaler/PCA fit on train only. `log_loss` and `hinge` SGD use
`alpha`, not the LR/LinearSVC `C` grid. API preflight runs before expensive work.

In [ ]:
from modules.config import load_config
from modules.epoch_training import validate_settings

config = load_config(repo_root / CONFIG_FILE)
if RUN_ID is not None:
    if Path(RUN_ID).name != str(RUN_ID):
        raise ValueError('RUN_ID must be a run name, not a path')
    family = 'epoch_runs' if ENGINE == 'sgd_epoch' else 'runs'
    run_dir = repo_root / config.get('results_dir', 'results') / family / RUN_ID
    if not (run_dir / 'config.json').is_file():
        raise FileNotFoundError(f'Run is absent at {run_dir}; restore its private backup first')
    config = load_config(run_dir / 'config.json')
    if ENGINE == 'sgd_epoch':
        EPOCH_SETTINGS = json.loads((run_dir / 'settings.json').read_text(encoding='utf-8'))
if ENGINE == 'sgd_epoch':
    EPOCH_SETTINGS = validate_settings(EPOCH_SETTINGS)
backup = None
if AUTO_BACKUP and ACTION != 'analyze':
    from modules.kaggle_backup import KaggleDatasetBackup
    backup = KaggleDatasetBackup(repo_root, BACKUP_DIR, BACKUP_DATASET, config=config)
print('Engine:', ENGINE, '| Action:', ACTION, '| Resume:', RUN_ID)
print('Remote backup:', 'enabled; each checkpoint must be verified' if backup else 'disabled for this action')
print('Epoch settings:', EPOCH_SETTINGS if ENGINE == 'sgd_epoch' else 'LR/LinearSVC C grid from config')

## 3. Data / frozen feature cache

Existing caches are verified against saved manifest/sample order/provenance and
byte hashes. A new classifier can use fixed cached features from an old GPU
without downloading weights or extracting again. Exact SGD resume additionally
requires its saved source and Python/serialization versions. Missing data is
downloaded only for preparation/extraction. Validation never scores test arrays.

In [ ]:
from modules.kaggle_workspace import prepare_features, saved_manifest

rows = None
feature_dirs = {}
manifest_sha = None
if ACTION == 'prepare':
    from modules.data import prepare_dataset, read_manifest
    prepared = prepare_dataset(repo_root, config, ARCHIVE_PATH)
    rows = read_manifest(prepared['manifest_path'])
    manifest_sha = prepared['manifest_sha256']
    if backup:
        backup('dataset_prepared')
elif ACTION in {'extract', 'train', 'final'}:
    extractors = ([EPOCH_SETTINGS['extractor']] if ENGINE == 'sgd_epoch' else
                  list(dict.fromkeys(e['extractor'] for e in config['experiments'])))
    rows, manifest_sha, feature_dirs = prepare_features(
        repo_root, config, extractors, archive_path=ARCHIVE_PATH, feature_dirs=FEATURE_DIRS,
        checkpoint_callback=backup, extract_missing=ACTION != 'final' and RUN_ID is None)
    print('Manifest checksum:', manifest_sha)
    print('Feature stores:', {k: str(v) for k, v in feature_dirs.items()})
else:
    print('Analysis/backup mode: no preparation, extraction or training')

In [ ]:
import pandas as pd
from IPython.display import display

if rows is not None:
    included = [r for r in rows if r['included']]
    frame = pd.DataFrame(included)
    display(frame.groupby('split').agg(files=('sample_id', 'count'), pixel_groups=('pixel_sha256', 'nunique')))
    display(frame.groupby(['label', 'split']).size().unstack(fill_value=0))
    if SHOW_PLOTS:
        import matplotlib.pyplot as plt
        frame[frame['split'] == 'train'].groupby('label').size().sort_values().plot.barh(figsize=(8, 7), title='Training files per family')
        plt.show()

## 4. Train / resume

Every SGD epoch saves an immutable state with preprocessing, optimizer counter,
shuffle RNG, history and best-epoch/patience state. Then private remote backup is
verified before the next epoch. A failure mid-epoch replays only that epoch.
`MAX_SESSION_MINUTES` pauses between epochs. Early stopping uses validation
representative Macro-F1; test is untouched. LR/LinearSVC retain trial-level resume.

In [ ]:
training_result = None
if ACTION == 'train':
    if ENGINE == 'sgd_epoch':
        from modules.epoch_training import run_epoch_training
        training_result = run_epoch_training(
            repo_root, config, rows, manifest_sha, feature_dirs, EPOCH_SETTINGS,
            resume_run=RUN_ID, checkpoint_callback=backup,
            max_session_seconds=MAX_SESSION_MINUTES * 60 if MAX_SESSION_MINUTES is not None else None,
            epochs_this_session=EPOCHS_THIS_SESSION)
        RUN_ID = training_result['run_id']
    else:
        from modules.experiments import run_validation
        training_result = run_validation(repo_root, config, rows, manifest_sha, feature_dirs,
                                         resume_run=RUN_ID, checkpoint_callback=backup)
        RUN_ID = Path(training_result['run_dir']).name
    print('Run ID:', RUN_ID)
    print('Status:', training_result['status'])
    print('Save this RUN_ID in controls before the next session.')
else:
    print('Training skipped for ACTION=', ACTION)

## 5. Analysis and validation evaluation

Compact tables replace the old full JSON dump. Selected-epoch validation metrics
are different from final held-out test metrics. Loss is unweighted training loss;
Macro-F1/accuracy are computed on one representative per exact-pixel group.

In [ ]:
from modules.kaggle_workspace import analysis_tables

tables = analysis_tables(repo_root, config, engine=ENGINE, run_id=RUN_ID)
for name, table in tables.items():
    print(name)
    display(table.head(30))
if SHOW_PLOTS and 'history' in tables and not tables['history'].empty:
    import matplotlib.pyplot as plt
    history = tables['history']
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    history.plot(x='epoch', y='train_loss_unweighted', ax=axes[0], title='Training objective')
    history.plot(x='epoch', y=['val_macro_f1', 'val_accuracy'], ax=axes[1], title='Validation representatives')
    plt.tight_layout()
    plt.show()
print('Inspect validation/errors.csv and per_class.csv; do not tune using final test scores.')

## 6. Explicit held-out test

After freezing the method and validation choice, set `ACTION='final'`,
`FINAL_TEST=True`, and the explicit completed `RUN_ID`, then run from controls.
This step never fits. A global final-test lock prevents later tuning or silently
repeating a failed/started test. Reopening a completed unchanged test reads saved
results. Keep validation and test results labeled separately in the report.

In [ ]:
final_result = None
if ACTION == 'final' and FINAL_TEST:
    if RUN_ID is None:
        raise ValueError('Choose the explicit completed RUN_ID before final test')
    if ENGINE == 'sgd_epoch':
        from modules.epoch_training import run_epoch_final_test
        final_result = run_epoch_final_test(repo_root, config, rows, manifest_sha, feature_dirs,
            EPOCH_SETTINGS, run_id=RUN_ID, confirm_frozen=True, checkpoint_callback=backup)
        metrics = final_result['metrics']['primary']
        print('FINAL TEST:', {k: metrics[k] for k in ('n_samples', 'accuracy', 'macro_f1')})
    else:
        from modules.experiments import run_final_test
        selection = repo_root / config.get('results_dir', 'results') / 'runs' / RUN_ID / 'selection.json'
        final_result = run_final_test(repo_root, config, rows, manifest_sha, selection)
        print('FINAL TEST status:', final_result['status'])
        if backup:
            backup('classical_final_completed', Path(training_result['run_dir']) if training_result else None)
else:
    print('Final test not opened. Keep FINAL_TEST=False while developing.')

## 7. Backup receipt / portable review export

The private Dataset's `recovery.zip` is the recovery payload; `latest.json` records
its checksums and run ID. Attach that Dataset (or use `RESTORE_REMOTE`) next time.
The review ZIP below omits features and is for reading/sharing results, not exact
recovery. Raw images/archive and credentials stay outside all backups.
Kaggle runtime quotas, network and Dataset version limits can change: verify
`REMOTE VERIFIED`, and retain earlier private versions. No automatic pruning.

In [ ]:
if ACTION == 'backup' and backup:
    backup('manual_checkpoint')
receipt = Path(BACKUP_DIR) / 'remote_receipt.json'
if receipt.is_file():
    saved = json.loads(receipt.read_text(encoding='utf-8'))
    print('Last verified remote receipt:', {k: saved.get(k) for k in ('dataset', 'label', 'run_id', 'remote_verified')})
if EXPORT_REVIEW_ZIP:
    from modules.submission import build_submission
    export_path = Path(BACKUP_DIR).parent / 'malimg-review-results.zip'
    build_submission(repo_root, export_path, draft=True, include_features=False)
    print('Review export:', export_path)
print('Workspace:', repo_root)
print('RUN_ID:', RUN_ID)